In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException   
from datetime import datetime
import traceback
import os
import pandas as pd
import re
import numpy as np
from time import sleep
import urllib.parse

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

In [6]:
# Set up functions
def print_update(func):
    def wrapper(*args, **kwargs):
        print(f'Running {func.__name__}')
        return func(*args, **kwargs)
    return wrapper


def reject_cookies(driver, temp=0, retries=2):
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.ID, 'onetrust-reject-all-handler')))
        driver.find_element(By.ID, 'onetrust-reject-all-handler').click()
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            print(f'{retries} attempts failed to Reject cookies')
        else:
            reject_cookies(driver, temp + 1)


def skip_tutorial(driver, temp=0, retries=2):
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip')))
        driver.find_element(By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip').click()
        driver.find_element(By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip').click()
        driver.find_element(By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip').click()
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            print(f'{temp} attempts failed to Skip the tutorial')
        else:
            skip_tutorial(driver, temp + 1)


def setup_driver():
    options = webdriver.ChromeOptions()
    options.set_capability("goog:loggingPrefs", {"performance": "INFO"})
    # prefs = {"profile.managed_default_content_settings.images": 2}
    # options.add_experimental_option("prefs", prefs)
    # options.add_argument("--headless")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--log-level=0")
    options.add_argument("--disable-extensions")
    

    driver = webdriver.Chrome(options=options)
    driver.maximize_window()

    driver.get('https://es.wallapop.com/app/search?filters_source=category_navigation&latitude=40.41956&longitude=-3.69196')

    reject_cookies(driver)
    skip_tutorial(driver)

    return driver


def load_more(driver, temp=0, retries=2):
    # Clicks button that initiates infinite scroll dynamic listings
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.ID, 'btn-load-more')))
        driver.find_element(By.ID, 'btn-load-more').click()
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            print(f'{temp} attempts failed to Click the load button')
        else:
            load_more(driver, temp + 1)


def wait_product(driver, temp=0, retries=3):
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'a.ItemCardList__item')))
        return True
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            print(f'{temp} attempts failed to Find listings')
        else:
            wait_product(driver, temp + 1)

In [7]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
import json

driver = setup_driver()
url = "https://es.wallapop.com/app/search?category_ids=24200&object_type_ids=9447&latitude=40.41956&longitude=-3.69196&filters_source=quick_filters&order_by=closest"
driver.get_log("performance")
driver.get(url)
wait_product(driver)
time.sleep(1)

In [13]:
logs = driver.get_log("performance")

In [14]:
import json

df = pd.DataFrame()
for i, log in enumerate(logs):
    log_json = json.loads(log["message"])["message"]
    if log_json["method"] == 'Network.requestWillBeSent':
        try:
            response_url = log_json["params"]["documentURL"]
            if response_url.startswith("https://api.wallapop.com/api/v3/search?"):
                request_id = log_json["params"]["initiator"]["requestId"]
                try:
                    response_body = driver.execute_cdp_cmd("Network.getResponseBody", {"requestId": request_id})
                    response_body_json = json.loads(response_body['body'])
                    json_df = pd.DataFrame(response_body_json['data']['section']['payload']['items'])
                    df = pd.concat([df, json_df])
                except Exception as e:
                    pass
        except KeyError:
            print(f'\n\n{log_json}')
df.head(5)

,id,user_id,title,description,category_id,price,images,reserved,location,shipping,favorited,bump,web_slug,created_at,modified_at,taxonomy,is_favoriteable,is_refurbished
0,9jdkeyvk196k,8j3y87ddy169,Apple iPhone 13 256GB Verde - Revisado,"Marca: Apple\nModelo: iPhone 13\nCapacidad de almacenamiento: 256 GB\nColor: Verde\nAccesorios: cable de carga \n\n🔋 Salud de la batería: +85% \n✅ Revisado: funciona correctamente \n🚫 Precio no negociable \n🚚 Envío gratis \n🔒 Factura y garantia de 1 año \n\nEl iPhone 13 reacondicionado es una excelente opción para aquellos que buscan un dispositivo de alta calidad a un precio más asequible. Este modelo ha sido restaurado y verificado por expertos para asegurar su buen funcionamiento, por lo que puedes estar seguro de que obtendrás un producto de calidad.\nEl iPhone 13 cuenta con una pantalla Retina de 6.1 pulgadas, con una resolución de 2532 x 1170 píxeles, lo que le permite mostrar imágenes y vídeos con una calidad impresionante. Además, cuenta con una cámara dual de 12 megapíxeles, que te permitirá capturar fotos y vídeos de alta calidad, incluso en condiciones de poca luz.\nEl iPhone 13 también cuenta con un procesador A15 Bionic, que te permitirá ejecutar varias tareas de manera rápida y fluida, incluso jugar juegos exigentes. Además, cuenta con una batería de larga duración que te permitirá usar tu dispositivo durante horas sin tener que preocuparte por recargarlo.\nEn resumen, el iPhone 13 reacondicionado es una excelente opción para aquellos que buscan un dispositivo de alta calidad a un precio asequible. Con su pantalla Retina de alta calidad, su cámara dual de 12 megapíxeles y su potente procesador A15 Bionic, este dispositivo te ofrecerá un rendimiento excepcional en una amplia variedad de tareas. Además, al ser reacondicionado, podrás ahorrar dinero al adquirirlo, sin sacrificar la calidad del producto.",24200,"{'amount': 483.0, 'currency': 'EUR'}","[{'average_color': '13C1AC', 'urls': {'small': 'https://cdn.wallapop.com/images/10420/h3/ef/__/c10420p1033637512/i5041733232.jpg?pictureSize=W320', 'medium': 'https://cdn.wallapop.com/images/10420/h3/ef/__/c10420p1033637512/i5041733232.jpg?pictureSize=W640', 'big': 'https://cdn.wallapop.com/images/10420/h3/ef/__/c10420p1033637512/i5041733232.jpg?pictureSize=W800'}}]",{'flag': False},"{'latitude': 40.42075510602411, 'longitude': -3.6887609899012093, 'postal_code': '28014', 'city': 'Madrid', 'region': 'Comunidad de Madrid', 'region2': 'Madrid', 'country_code': 'ES'}","{'item_is_shippable': True, 'user_allows_shipping': True, 'cost_configuration_id': '04cf65ea-42f5-11ed-b878-0242ac120002'}",{'flag': False},{'type': 'country'},apple-iphone-13-256gb-verde-revisado-1033637512,1722340466985,1722943742164,"[{'id': 24200, 'name': 'Tecnología y electrónica', 'icon': 'robot'}, {'id': 24201, 'name': 'Telefonía: móviles y smartwatches', 'icon': None}, {'id': 9447, 'name': 'Smartphones', 'icon': None}]",{'flag': True},{'flag': True}
1,e6589ne8dg6o,8j3y87ddy169,Apple iPhone 13 128GB Verde - Revisado,"Marca: Apple\nModelo: iPhone 13\nCapacidad de almacenamiento: 128 GB\nColor: Verde\nAccesorios: cable de carga \n\n🔋 Salud de la batería: +85% \n✅ Revisado: funciona correctamente \n🚫 Precio no negociable \n🚚 Envío gratis \n🔒 Factura y garantia de 1 año \n\nEl iPhone 13 reacondicionado es una excelente opción para aquellos que buscan un dispositivo de alta calidad a un precio más asequible. Este modelo ha sido restaurado y verificado por expertos para asegurar su buen funcionamiento, por lo que puedes estar seguro de que obtendrás un producto de calidad.\nEl iPhone 13 cuenta con una pantalla Retina de 6.1 pulgadas, con una resolución de 2532 x 1170 píxeles, lo que le permite mostrar imágenes y vídeos con una calidad impresionante. Además, cuenta con una cámara dual de 12 megapíxeles, que te permitirá capturar fotos y vídeos de alta calidad, incluso en condiciones de poca luz.\nEl iPhone 13 también cuenta con un procesador A15 Bionic, que te permitirá ejecut

In [63]:
import requests

api_url = "https://api.wallapop.com/api/v3/search?category_id=12800&subcategory_ids=10164&latitude=40.41956&longitude=-3.69196&source=quick_filters&show_multiple_sections=false" # first page


data = requests.get(api_url)
print(data.json())


{'data': {'section': {'payload': {'order': 'closest', 'title': 'Find what you want', 'items': [{'id': 'ejk8d82dvpjx', 'user_id': '4z4vq145nmzy', 'title': 'INTERCOMUNICADOR CON CAMARA INCORPORADA FREEDCONN', 'description': 'INTERCOMUNICADOR FREEDCONN R3, CON CAMARA INCORPORADA PODRAS GRABAR HACIA DELANTE O HACIA ATRAS, CARGA RAPIDA, 6 O 7 HORAS DE GRABACION EN 2K, BLUETOOTH 5.0 CON EL CHIP MAS RAPIDO QUE HAY EN EL MERCADO, HASTA 6 PILOTOS EN CONVERSACION CON 1000 METROS DE DISTANCIA, REDUCCION DE RUIDO HASTA UN 95% MENOS, RADIO FM INCORPORADA, FUNCION WIFI, IMPERMEABLE CON CERTIFICACION IP67, ALTAVOCES HQ SONY, NUESTRO MEJOR INTERCOMUNICADOR SIN DUDA, 2 AÑOS DE GARANTIA OFICIAL', 'category_id': 12800, 'price': {'amount': 169.0, 'currency': 'EUR'}, 'images': [{'average_color': '13C1AC', 'urls': {'small': 'https://cdn.wallapop.com/images/10420/h5/ac/__/c10420p1036806271/i5061502513.jpg?pictureSize=W320', 'medium': 'https://cdn.wallapop.com/images/10420/h5/ac/__/c10420p1036806271/i50615025

In [4]:
directory = 'data/listings/'
dfs = []
for filename in os.listdir(directory):
    if filename.endswith(".csv"):
        filepath = os.path.join(directory, filename)
        df = pd.read_csv(filepath)
        dfs.append(df)
listings = pd.concat(dfs)
listings.to_csv('data/listings.csv', index=False)
listings

FileNotFoundError: [Errno 2] No such file or directory: 'data/listings/'